# Embedding Model Selection & Hyperparameter Grid Search

**Latest version:** May 2026

## Notebook overview

This notebook is preparing everything needed before the actual BERTopic topic modelling run (which lives in `3_topic_modelling.ipynb`).

| Step | What happens | Output |
|------|-------------|--------|
| 1 | Imports & helper functions | - |
| 2 | Load documents | - |
| 3 | Compare transformer models | `embed_test_results.pkl` |
| 4 | Encode the full corpus | `final_embeddings.npy` |
| 5 | UMAP/HDBSCAN grid search | `gridsearch_results.csv`, `best_params.json` |


**⚠️ Runtime note:** Runtime can vary depending on your input data and hardware. Each runtime-demanding cell saves intermediate checkpoints so they can be safely interrupted and resumed.  


**🔁 Reproducibility note:** Results may vary slightly across runs and hardware. Set your seed to `random_state=42` throughout to minimise variance.

---
# 1. Imports & helper functions

In [1]:
from config import *

import gc
import json
import os
import pickle
import requests
import time
from itertools import product
from pathlib import Path
from time import sleep
import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

# Environment optimization: reduce thread contention between native libraries
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]       = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Helper functions
- `get_local_model_path`: construct the local cache path for a Hugging Face model that has already been downloaded.
- `safe_load_checkpoint`: loads a pickle checkpoint robustly.
- `compute_topic_diversity`: calculate the ratio of unique words to total word slots across all top-*k* topic representations (adapted from [here](https://octis.readthedocs.io/en/latest/_modules/octis/evaluation_metrics/diversity_metrics.html?utm_source=chatgpt.com)).
- `compute_npmi_coherence`: mean Normalised Pointwise Mutual Information (NPMI) across all topic word pairs to capture semantic quality (topic coherence).

In [2]:
def get_local_model_path(repo_id: str) -> Path:

    cache_base = Path.home() / ".cache" / "huggingface" / "hub"
    cache_name = f"models--{repo_id.replace('/', '--')}"
    return cache_base / cache_name / "snapshots" / "main"


def safe_load_checkpoint(checkpoint_file: str | Path):
    if not os.path.exists(checkpoint_file):
        return None
    try:
        with open(checkpoint_file, "rb") as f:
            return pickle.load(f)
    except (EOFError, pickle.UnpicklingError):
        print("⚠️  Corrupted checkpoint detected – starting fresh.")
        os.remove(checkpoint_file)
        return None


def compute_topic_diversity(topic_model: BERTopic, topk: int = 10) -> float:
    topics = topic_model.get_topics()
    all_words = [
        word
        for topic_id, topic_words in topics.items()
        if topic_id != -1
        for word, _ in topic_words[:topk]
    ]
    return len(set(all_words)) / len(all_words) if all_words else 0.0


def compute_npmi_coherence(
    topic_model: BERTopic,
    docs: list,
    topk: int = 10,
    eps: float = 1e-12,
) -> float:
    topics_dict = topic_model.get_topics()
    n_docs = len(docs)

    # Get topic words
    vocab = set()
    topic_word_lists = {}
    for tid, twords in topics_dict.items():
        if tid == -1:   # Exclude outlier topic
            continue
        words = [w for w, _ in twords[:topk]]
        topic_word_lists[tid] = words
        vocab.update(words)

    if not vocab or not topic_word_lists:
        return 0.0

    # Document-level word presence matrix
    doc_sets = [set(d.lower().split()) for d in docs]

    # Marginal probabilities  P(w)
    p_word = {
        w: sum(1 for ds in doc_sets if w in ds) / n_docs
        for w in vocab
    }

    # Joint probabilities  P(w_i, w_j)  for all required pairs
    def p_joint(wi: str, wj: str) -> float:
        return sum(1 for ds in doc_sets if wi in ds and wj in ds) / n_docs

    # Mean NPMI across all topic word-pair combinations
    topic_scores = []
    for words in topic_word_lists.values():
        pair_scores = []
        for i in range(len(words)):
            for j in range(i + 1, len(words)):
                wi, wj = words[i], words[j]
                pw_i = p_word.get(wi, 0.0)
                pw_j = p_word.get(wj, 0.0)
                pw_ij = p_joint(wi, wj)
                if pw_i < eps or pw_j < eps or pw_ij < eps:
                    continue
                pmi  = np.log(pw_ij / (pw_i * pw_j + eps) + eps)
                npmi = pmi / (-np.log(pw_ij + eps))
                pair_scores.append(npmi)
        if pair_scores:
            topic_scores.append(float(np.mean(pair_scores)))

    return float(np.mean(topic_scores)) if topic_scores else 0.0

---
# 2. Load documents

In [ ]:
print("Loading preprocessed documents...")
with open(PREPROCESSED_DOCS_PATH, "rb") as f:
    preprocessed_docs = pickle.load(f)

# Create index
ids = list(range(len(preprocessed_docs)))

print(f"Loaded {len(preprocessed_docs):,} preprocessed documents")

## Data quality checks

Before modelling, verify that the loaded data is clean and properly aligned.  

Checks covered:
1. **Alignment**: IDs and docs have the same length.
2. **Nulls / empty strings**: documents with no text would embed to a near-zero vector and inflate the outlier topic.
3. **Duplicates**: identical documents embed identically and can artificially inflate certain clusters.
4. **Length distribution**: very short documents (< 2 words) are  flagged.

In [ ]:
# 1. Alignment
assert len(ids) == len(preprocessed_docs), "❌ ID/Doc length mismatch!"
print(f"✅ Alignment OK  – {len(preprocessed_docs):,} docs & IDs")

# 2. Nulls / empty strings
null_count  = sum(1 for d in preprocessed_docs if d is None)
empty_count = sum(1 for d in preprocessed_docs if isinstance(d, str) and d.strip() == "")
print(f"   Null docs   : {null_count:,}")
print(f"   Empty docs  : {empty_count:,}")

# 3. Duplicates
n_dupes = len(preprocessed_docs) - len(set(preprocessed_docs))
print(f"   Duplicates  : {n_dupes:,} ({n_dupes/len(preprocessed_docs)*100:.2f}%)")

# 4. Length distribution (word count)
lengths = [len(d.split()) for d in preprocessed_docs if isinstance(d, str)]
print(f"\nDocument length (words):")
print(f"   min={min(lengths)}, median={int(np.median(lengths))}, "
      f"mean={np.mean(lengths):.1f}, max={max(lengths)}")
short_docs = sum(1 for l in lengths if l < 2)
print(f"   Docs with < 2 words: {short_docs:,} ({short_docs/len(lengths)*100:.1f}%)")

## Create a random sub-sample (if needed)

In case you have a large dataset (> 1. million), a fixed random sample can be used for:
- Step 0: embedding model comparison  
- Step 2: sensitivity analysis & grid search  

#### Why use a random sample?

Training BERTopic on millions of documents is computationally expensive and can take a long time. A single UMAP + HDBSCAN fit on millions of documents can take hours/days and requires large amounts of RAM and/or GPU memory. Using a random sample during the exploratory phases (model comparison, sensitivity analysis, grid search) offers several advantages:

**Speed.** Fitting and evaluating dozens of hyperparameter combinations on a sub-sample of documents reduces the runtime from hours to minutes per run, making iterative experimentation feasible.

**Reproducibility.** A fixed random seed (`np.random.seed(42)`) ensures the same documents are sampled every time, so results are directly comparable across runs and researchers.

**Sufficient signal.** Topic structures in large corpora are typically stable - the dominant themes present in millions of documents can already be well-represented in a 50,000-document sample. 

**Resource efficiency.** Grid search and sensitivity analysis are diagnostic steps, not final outputs. Spending compute on the full corpus at this stage is wasteful when the goal is parameter selection rather than final topic assignments.

In [ ]:
# Set your desired sample size here. 
# If the corpus is smaller than this value, the full corpus is used automatically, but still run this cell.
DESIRED_SAMPLE_SIZE = 50_000

SAMPLE_SIZE = min(DESIRED_SAMPLE_SIZE, len(preprocessed_docs))
np.random.seed(42)

sample_idx  = np.random.choice(len(preprocessed_docs), SAMPLE_SIZE, replace=False)
docs_sample = [preprocessed_docs[i] for i in sample_idx]

print(f"Desired sample size : {DESIRED_SAMPLE_SIZE:,}")
print(f"Actual sample size  : {SAMPLE_SIZE:,} ({'full corpus' if SAMPLE_SIZE == len(preprocessed_docs) else 'random sample'})")
print(f"Sample coverage     : {SAMPLE_SIZE/len(preprocessed_docs)*100:.2f}% of corpus")

---
# 3. Embedding model comparison
This pipeline makes use of [word embeddings](https://cssbook.net/content/chapter10.html#sec-wordembeddings) to represent text as data. If you wish to use a different representation of text as data, you can change this here.

Run this section to pick the best sentence-transformer for your corpus. The comparison evaluates each model on the data or sample using BERTopic with fixed, default UMAP/HDBSCAN settings, so differences in metrics are driven only by embedding quality.

Metrics reported:
- [**Silhouette score**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html): a quantitative metric for how well clusters are separated in embedding space (excludes outlier topic -1)
- **Topic diversity**: lexical uniqueness across top-10 keywords per topic
- **Outlier proportion**: % share of documents assigned to topic -1
- **Runtime**: how long each model takes.

Change below to which models you would like to test.

In [6]:
TEST_MODELS = [
    # Baseline models
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-MiniLM-L12-v2",
    "sentence-transformers/all-mpnet-base-v2",
    # Specialised for short text / social media
    "sentence-transformers/paraphrase-MiniLM-L6-v2",
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    # Multilingual
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    # Modern compact models
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5",
    # E5 family
    "intfloat/e5-small-v2",
    "intfloat/e5-base-v2",
]

---
### Download models (only needs to be run once)

The cell below downloads all models to the HuggingFace cache. Skip if already cached.

In [ ]:
def download_model_files(repo_id: str, cache_base=None, verbose: bool = False, max_retries: int = 3) -> bool:
    if cache_base is None:
        cache_base = Path.home() / ".cache" / "huggingface" / "hub"

    snapshot_dir = cache_base / f"models--{repo_id.replace('/', '--')}" / "snapshots" / "main"
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    base_url = f"https://huggingface.co/{repo_id}/resolve/main/"

    base_files = [
        "model.safetensors", "pytorch_model.bin", "config.json",
        "tokenizer.json", "tokenizer_config.json", "vocab.txt",
        "special_tokens_map.json", "sentence_bert_config.json",
        "config_sentence_transformers.json", "modules.json",
    ]

    success, failed_files = 0, []

    def _download(url: str, out: Path) -> bool:
        for attempt in range(max_retries):
            try:
                with requests.get(url, stream=True, timeout=120) as r:
                    if r.status_code == 404:
                        return False
                    r.raise_for_status()
                    with open(out, "wb") as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                return True
            except Exception as e:
                if verbose:
                    print(f"  ⚠️  Attempt {attempt + 1} failed for {out.name}: {e}")
                sleep(5)
        return False

    for filename in base_files:
        out = snapshot_dir / filename
        if out.exists() and out.stat().st_size > 0:
            success += 1
            continue
        if _download(base_url + filename, out):
            success += 1
        else:
            failed_files.append(filename)

    for i in range(6):
        for mod in ["Transformer", "Pooling", "Dense", "Normalization"]:
            subdir = snapshot_dir / f"{i}_{mod}"
            subdir.mkdir(exist_ok=True)
            cfg = subdir / "config.json"
            if not cfg.exists():
                _download(f"{base_url}{i}_{mod}/config.json", cfg)

    if verbose:
        print(f"✅  {repo_id}: {success} files cached")
        if failed_files:
            print(f"   ⚠️  Failed: {failed_files}")
    return success > 0


# Bulk download
print("🚀  Starting bulk model download...")
cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
print(f"Nr. of models to download: {len(TEST_MODELS)}\n")

successful, failed = 0, []
for model in tqdm(TEST_MODELS, desc="📦 Downloading models"):
    try:
        ok = download_model_files(model, verbose=False)
        successful += ok
        if not ok:
            failed.append(model)
    except KeyboardInterrupt:
        print("\n⚠️  Interrupted.")
        break
    except Exception:
        failed.append(model)

print(f"\n{'='*60}")
print(f"✅  Successful: {successful}/{len(TEST_MODELS)}")
if failed:
    print(f"❌  Failed ({len(failed)}):")
    for m in failed:
        print(f"   - {m}")

---
### Compare Sentence-Transformer Models

[UMAP and HDBSCAN parameters](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html) are held constant/default across models so that evaluation metrics (e.g., silhouette score, topic diversity, and outlier rate) reflect **embedding quality only**.

#### Parameters

| Parameter | Type | Description |
|---|---|---|
| `docs` | `list[str]` | Documents to embed (entire data or defined sample).|
| `model_names` | `list[str]` | Hugging Face models to evaluate. |
| `batch_size` | `int` | Batch size used during encoding. |
| `verbose` | `bool` | Whether to print progress updates and sample topics. |
| `umap_random_state` | `int` | Fixed random seed for UMAP to ensure comparable projections across models. |

#### Returns

| Type | Description |
|---|---|
| `pd.DataFrame` | Results sorted by silhouette score in descending order. |

**⚠️ GPU recommended.** Runtime varies depending on hardware.

In [ ]:
def evaluate_sentence_transformers(
    docs: list,
    model_names: list,
    batch_size: int = 128,
    verbose: bool = True,
    umap_random_state: int = 42,
) -> pd.DataFrame:

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if verbose:
        print(f"Using device: {device}")

    # Using default UMAP/HDBSCAN values – kept constant so only embedding quality varies
    umap_model    = UMAP(random_state=umap_random_state, n_components=10, metric="cosine")
    hdbscan_model = HDBSCAN(min_cluster_size=10, prediction_data=True)

    results = []
    for model_name in model_names:
        if verbose:
            if torch.cuda.is_available():
                print(f"\nGPU memory before: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print(f"\n🔹  Evaluating {model_name}...")

        start = time.time()

        # Load model (prefer local cache)
        try:
            local = get_local_model_path(model_name)
            load_path = str(local) if local.exists() else model_name
            model = SentenceTransformer(load_path, device=device)
        except Exception as e:
            print(f"⚠️  Error loading {model_name}, falling back to CPU: {e}")
            model = SentenceTransformer(model_name, device="cpu")

        embeddings = model.encode(
            docs, batch_size=batch_size, show_progress_bar=verbose, device=device
        )

        topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            embedding_model=None,
            verbose=False,
        )
        topics, _ = topic_model.fit_transform(docs, embeddings)

        # Metrics (outliers excluded from silhouette)
        topics_arr      = np.array(topics)
        non_outlier_mask = topics_arr != -1
        emb_filtered    = embeddings[non_outlier_mask]
        top_filtered    = topics_arr[non_outlier_mask]

        n_topics          = len(set(top_filtered))
        silhouette        = silhouette_score(emb_filtered, top_filtered) if n_topics > 1 else -1
        diversity         = compute_topic_diversity(topic_model)
        outlier_prop      = round((~non_outlier_mask).sum() / len(topics), 4)
        duration          = round(time.time() - start, 2)

        if verbose:
            info = topic_model.get_topic_info()
            sample_topics = info[info["Topic"] != -1].head(5)
            print(f"\n  📋  Sample topics:")
            for _, row in sample_topics.iterrows():
                words = ", ".join(w for w, _ in topic_model.get_topic(row["Topic"])[:8])
                print(f"      Topic {row['Topic']:>3} ({row['Count']:>5} docs): {words}")
            print(
                f"\n  ✅  {model_name} | topics={n_topics}, silhouette={silhouette:.4f}, "
                f"diversity={diversity:.4f}, outliers={outlier_prop:.1%}, time={duration}s"
            )

        results.append({
            "model": model_name,
            "num_topics": n_topics,
            "silhouette": silhouette,
            "topic_diversity": diversity,
            "outlier_proportion": outlier_prop,
            "time_seconds": duration,
        })

        # Free GPU memory before next model
        del model, embeddings, topic_model
        torch.cuda.empty_cache()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.ipc_collect()

    return (
    pd.DataFrame(results)
    .assign(viable=lambda df: df["num_topics"] >= 30)
    .sort_values(by=["viable", "silhouette"], ascending=[False, False])
    )

In [ ]:
# ⚠️ Runtime warning. Comment out if transformer model is already chosen.
embedtest_results_df = evaluate_sentence_transformers(docs_sample, TEST_MODELS, batch_size=128)

In [ ]:
print("\nEmbedding model comparison results:")
embedtest_results_df

In [ ]:
# Save results so Step 1 can reload without re-running
embedtest_results_df.to_pickle(EMBED_TEST_PATH)
print(f"✅ Results saved.")

---
# 4. Create Full-Corpus Embeddings

Using the best-performing model from Step 3, we encode all documents in the dataset.

The function below supports **incremental checkpointing**, allowing long embedding jobs to be safely interrupted and resumed. If a completed `final_embeddings.npy` file already exists in the checkpoint directory, it is loaded immediately. Otherwise, the function resumes from the latest checkpoint and continues encoding.

During processing:
- Intermediate checkpoints are saved after a configurable number of documents.
- Once encoding is complete, all embeddings are merged into a single `.npy` file.
- Temporary checkpoint files are automatically removed after successful completion.

### GPU Optimizations

Several optimizations are enabled to improve speed and reduce memory usage:

- **`use_fp16=True`**  
  Uses 16-bit precision on compatible GPUs, reducing memory usage by ~50% and often speeding up inference.

- **`torch.compile`**  
  Enables PyTorch 2.0+ graph optimization for additional inference speedups (silently skipped if unsupported).

- **Memory pre-allocation**  
  For corpora smaller than ~10 million documents, a single NumPy array is pre-allocated to avoid repeated concatenation overhead.


### Parameters

| Parameter | Description |
|---|---|
| `docs` | List of text documents to encode. |
| `model_name` | SentenceTransformer model name or Hugging Face repository ID. |
| `batch_size` | Encoding batch size (larger batches are faster but require more memory). |
| `checkpoint_every` | Save a checkpoint after this many processed documents. |

**⚠️ Runtime Warning: Embedding large corpora can take a long time depending on dataset size and available hardware. GPU acceleration is strongly recommended for large-scale encoding tasks.**

In [ ]:
# Reload Step 3 to avoid re-running the evaluation
embedtest_results_df = pd.read_pickle(EMBED_TEST_PATH)
best_model_name = embedtest_results_df.iloc[0]["model"]
print(f"🏆  Best embedding model: {best_model_name}")

In [16]:
def encode_with_checkpointing(
    docs: list,
    model_name: str,
    batch_size: int = 512,
    checkpoint_dir=CHECKPOINT_DIR,
    embeddings_dir=EMBEDDINGS_DIR, # 
    checkpoint_every: int = 5000, # save every 5000 documents
    device=None, # detect GPU/CPU automatically
    use_fp16: bool = True, # only available with CPU
) -> np.ndarray:

    # Checkpoint directory
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_file = os.path.join(checkpoint_dir, "checkpoint_embeddings.pkl")

    # Final embeddings directory
    os.makedirs(embeddings_dir, exist_ok=True)
    model_safe_name = model_name.replace("/", "_")
    final_file = os.path.join(
        embeddings_dir,
        f"final_embeddings.npy"
    )

    if os.path.exists(final_file):
        print("✅  Final embeddings already exist – loading from disk.")
        return np.load(final_file)

    # Detect device
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load model
    try:
        local = get_local_model_path(model_name)
        embedder = SentenceTransformer(
            str(local) if local.exists() else model_name, device=device
        )
    except Exception as e:
        print(f"⚠️  Error – falling back to CPU: {e}")
        embedder = SentenceTransformer(model_name, device="cpu")

    if device == "cuda" and use_fp16:
        embedder.half()
        print("FP16 mixed precision enabled")

    if hasattr(torch, "compile") and device == "cuda":
        try:
            embedder = torch.compile(embedder)
            print("Model compiled with PyTorch 2.0+ optimisation")
        except Exception:
            pass

    # Resume from checkpoint if available
    checkpoint = safe_load_checkpoint(checkpoint_file)
    start_idx, embeddings_list, all_embeddings = 0, [], None
    if checkpoint:
        start_idx       = checkpoint.get("last_idx", 0)
        embeddings_list = checkpoint.get("embeddings", [])
        if checkpoint.get("partial_array") is not None:
            all_embeddings = checkpoint["partial_array"]
        print(f"Resuming from document {start_idx:,}/{len(docs):,}")

    # Pre-allocate a single array if memory allows
    use_preallocated = False
    if all_embeddings is None and len(docs) < 10_000_000:
        try:
            dim = embedder.get_sentence_embedding_dimension()
            all_embeddings = np.empty((len(docs), dim), dtype=np.float32)
            use_preallocated = True
            print(f"Pre-allocated array: {all_embeddings.shape}")
        except Exception:
            print("Memory pre-allocation skipped (insufficient RAM).")

    # Main encoding loop
    try:
        batches = range(start_idx, len(docs), batch_size)
        pbar = tqdm(
            batches,
            desc="Encoding",
            initial=start_idx // batch_size,
            total=(len(docs) + batch_size - 1) // batch_size,
        )
        for i in pbar:
            batch = docs[i : min(i + batch_size, len(docs))]
            with torch.no_grad():
                batch_emb = embedder.encode(
                    batch,
                    batch_size=batch_size,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=False,  # UMAP handles normalisation
                    device=device,
                )

            if use_preallocated:
                end = min(i + batch_size, len(docs))
                all_embeddings[i:end] = batch_emb
            else:
                embeddings_list.append(batch_emb)

            if i > start_idx and pbar.format_dict["elapsed"] > 0:
                docs_per_sec = (i - start_idx) / pbar.format_dict["elapsed"]
                pbar.set_postfix({"docs/s": f"{docs_per_sec:.0f}"})

            # Save checkpoint periodically
            if (i + batch_size - start_idx) % checkpoint_every == 0 or i + batch_size >= len(docs):
                end = min(i + batch_size, len(docs))
                ckpt = {
                    "last_idx": end,
                    "embeddings":    embeddings_list if not use_preallocated else None,
                    "partial_array": all_embeddings[:end] if use_preallocated else None,
                }
                with open(checkpoint_file, "wb") as f:
                    pickle.dump(ckpt, f, protocol=pickle.HIGHEST_PROTOCOL)

    # Handle interruptions
    except KeyboardInterrupt:
        print("\n⏸  Interrupted – saving progress...")
        ckpt = {
            "last_idx":      i,
            "embeddings":    embeddings_list if not use_preallocated else None,
            "partial_array": all_embeddings[:i] if use_preallocated else None,
        }
        with open(checkpoint_file, "wb") as f:
            pickle.dump(ckpt, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Checkpoint saved at document {i:,}. Re-run to resume.")
        raise

    embeddings = all_embeddings if use_preallocated else np.vstack(embeddings_list)
    np.save(final_file, embeddings)
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"✅ Encoding complete! Shape: {embeddings.shape}")
    return embeddings

In [ ]:
# ⚠️ Runtime warning (up to ~9 h on GPU on 6 M very short docs). Checkpoints are saved automatically.
embeddings = encode_with_checkpointing(preprocessed_docs, model_name=best_model_name)

In [ ]:
# Sanity check on the final embedding array

# Do the dimensions (docs & embedding model) add up with the final embeddings?
# The second number depends on the embedding model, e.g. Qwen3-VL-Embedding-8B outputs 4096 dimensions - check documentation for the model you selected.
print(f"Embedding shape  : {embeddings.shape}")

# How many videos were actually emedded?
print(f"Nr of docs       : {embeddings.shape[0]} (should match {len(preprocessed_docs)})")

# Did every document get an embedding? (Must be True)
print(f"Matches docs     : {embeddings.shape[0] == len(preprocessed_docs)}")

# Did any batch fail silently? (Must be False)
print(f"Contains NaN     : {np.isnan(embeddings).any()}")

# Did any value overflow during embedding? (Must be False)
print(f"Contains Inf     : {np.isinf(embeddings).any()}")

# Was the full array written to the disk correctly?
print(f"File size on disk: {os.path.getsize(os.path.join(EMBEDDINGS_DIR, 'final_embeddings.npy')) / 1e9:.2f} GB")

---
# 5. Hyperparameter grid search

A grid search is used to identify the best UMAP and HDBSCAN hyperparameters on the data/sample before scaling to the full corpus.

In [ ]:
# Reload embeddings (memory-mapped to avoid loading the full array twice)
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
print(f"Embeddings shape : {embeddings.shape}")

# If using a sample, extract the sample slice
embeddings_sample = embeddings[sample_idx].copy()
assert len(docs_sample) == len(embeddings_sample), "Sample size mismatch!"
print(f"Sample embeddings: {embeddings_sample.shape}  ({embeddings_sample.nbytes / 1e9:.2f} GB)")

## Optional: Sample-size sensitivity analysis

**📍 Note:** This section can be skipped if you use the entire sample for gridsearch.

Before committing to a training sample for the grid search, we verify that results are stable across sample sizes. The analysis fits BERTopic to differently-sized samples and reports quantitative metrics and sample topics.

In [ ]:
# Check whether a sample is used for grid search
print(f"Using full corpus  : {SAMPLE_SIZE == len(preprocessed_docs)}")
print(f"Sample size        : {SAMPLE_SIZE:,} / {len(preprocessed_docs):,} docs ({SAMPLE_SIZE/len(preprocessed_docs)*100:.2f}%)")

In [ ]:
# Compute silhouette score on a random sub-sample, excluding outlier topic -1.
def _silhouette_on_sample(
    embeddings: np.ndarray,
    topics: np.ndarray,
    sample_size: int = 50_000, # Adjust based on your data/preference.
    random_seed: int = 42,
) -> float:

    rng  = np.random.default_rng(random_seed)
    idx  = rng.choice(len(topics), size=min(sample_size, len(topics)), replace=False)
    t, e = topics[idx], embeddings[idx]
    mask = t != -1
    t, e = t[mask], e[mask]
    if len(np.unique(t)) < 2 or len(t) < 2:
        return -1.0
    return float(silhouette_score(e, t))

In [ ]:
def run_sample_sensitivity(
    docs: list,
    embeddings: np.ndarray,
    sample_sizes: list,
    umap_params: dict,
    hdbscan_params: dict,
    silhouette_sample: int = 50_000, # Adjust based on your data/preferences.
    n_sample_topics: int = 5,
    n_sample_docs: int = 3,
    random_seed: int = 42,
    save_path=None,
) -> pd.DataFrame:

    rng, results = np.random.default_rng(random_seed), []
    reference_labels: set | None = None  # top-5 labels from the reference run

    for n in sample_sizes:
        print(f"\n{'='*60}")
        print(f"Sample size: {n:,}")
        print(f"{'='*60}")

        idx    = rng.choice(len(docs), size=min(n, len(docs)), replace=False)
        s_docs = [docs[i] for i in idx]
        s_embs = embeddings[idx]

        # Topic model on sample / default settings
        tm = BERTopic(
            embedding_model=None,
            umap_model=UMAP(
                **umap_params, metric="cosine", random_state=random_seed,
                low_memory=True, n_jobs=-1,
            ),
            hdbscan_model=HDBSCAN(
                **hdbscan_params, metric="euclidean",
                cluster_selection_method="eom", core_dist_n_jobs=-1,
                prediction_data=False,
            ),
            vectorizer_model=CountVectorizer(
                ngram_range=(1, 1), stop_words="english",
                max_features=10_000, min_df=1,
            ),
            calculate_probabilities=False,
            verbose=False,
        )

        t0 = time.time()
        topics, _ = tm.fit_transform(s_docs, s_embs)
        elapsed   = time.time() - t0
        topics_arr = np.array(topics)

        # Quantitative metrics
        n_topics    = len(np.unique(topics_arr[topics_arr != -1]))
        outlier_pct = (topics_arr == -1).mean() * 100
        sil         = _silhouette_on_sample(s_embs, topics_arr, silhouette_sample, random_seed)
        diversity   = compute_topic_diversity(tm)

        print(f"  Topics      : {n_topics}")
        print(f"  Outliers    : {outlier_pct:.1f}%")
        print(f"  Silhouette  : {sil:.4f}")
        print(f"  Diversity   : {diversity:.4f}")
        print(f"  Time        : {elapsed:.1f}s  ({n/elapsed:.0f} docs/s)")

        # Qualitative inspection
        info = tm.get_topic_info()
        info = info[info["Topic"] != -1].head(n_sample_topics)
        current_labels = set(info["Name"].tolist()) if "Name" in info.columns else set()

        print(f"\n  Sample topics (top {n_sample_topics} by size):")
        print(f"  {'-'*55}")
        for _, row in info.iterrows():
            words   = ", ".join(w for w, _ in tm.get_topic(row["Topic"])[:8])
            doc_idx = np.where(topics_arr == row["Topic"])[0]
            samples = [s_docs[i] for i in doc_idx[:n_sample_docs]]
            print(f"  Topic {row['Topic']:>3} ({row['Count']:>5} docs): {words}")
            for doc in samples:
                print(f"    → {doc}")

        # # OPTIONAL. CHANGE VALUES AS DESIRED. # #
        # The reference run sets the reference; all subsequent sizes are compared to it.
        # Topic overlap vs reference run
        
        # Change values here to determine reference sample size:
        if n == 50_000 or (reference_labels is None and n >= 50_000): 
            reference_labels = current_labels
            overlap = 1.0  # the reference run overlaps with itself by definition
        elif reference_labels is not None and current_labels:
            overlap = len(current_labels & reference_labels) / len(current_labels)
        else:
            overlap = float("nan")

        print(f"  Topic overlap vs reference run: {overlap:.2f}" if not np.isnan(overlap) else "  Topic overlap vs reference run: N/A (reference not yet set)")

        results.append({
            "sample_size":          n,
            "n_topics":             n_topics,
            "outlier_pct":          round(outlier_pct, 2),
            "silhouette":           round(sil, 4),
            "diversity":            round(diversity, 4),
            "topic_overlap_vs_sample": round(overlap, 4) if not np.isnan(overlap) else None,
            "time_seconds":         round(elapsed, 1),
            "docs_per_sec":         round(n / elapsed, 0),
        })

        # Clear memory
        del tm
        gc.collect()

    df = pd.DataFrame(results)
    if save_path:
        df.to_csv(save_path, index=False)
        print(f"\n💾 Results aved.")
    return df

In [ ]:
# Fixed BERTopic settings used for sensitivity analysis only
SENSITIVITY_UMAP_PARAMS    = {"n_neighbors": 15, "n_components": 10, "init": "random"}
SENSITIVITY_HDBSCAN_PARAMS = {"min_cluster_size": 10}

# Sample sizes to test, change as desired
SAMPLE_SIZES = [25_000, 50_000, 75_000, 100_000, 200_000]

**What to look out for:**
- Topic count should increase with sample size (more data → more discovered topics).
- Silhouette score and outlier rate should be broadly stable once the sample is large enough to represent the main communities.
- Topic labels should be qualitatively consistent across sizes.
- **`topic_overlap_vs_sample`** – fraction of the current run's top-5 topic labels that also appeared in the reference run. A value consistently above 0.6 indicates the reference sample already captures the dominant communities and is a valid grid-search training size. A value that drops sharply at larger sizes suggests important topics only emerge with more data.

In [ ]:
# ⚠️ Runtime scales with sample sizes
sensitivity_df = run_sample_sensitivity(
    docs=preprocessed_docs,
    embeddings=embeddings,
    sample_sizes=SAMPLE_SIZES,
    umap_params=SENSITIVITY_UMAP_PARAMS,
    hdbscan_params=SENSITIVITY_HDBSCAN_PARAMS,
    save_path=SAMPLE_SENSITIVITY_PATH,
)
sensitivity_df

In [ ]:
# Visualise metrics across sample sizes
# sensitivity_df = pd.read_csv(SAMPLE_SENSITIVITY_PATH)  # reload if needed

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title, color in [
    (axes[0], "n_topics",    "Number of topics",  "#4A90E2"),
    (axes[1], "outlier_pct", "Outlier rate (%)",  "#E87040"),
    (axes[2], "silhouette",  "Silhouette score",  "#5DAD5D"),
]:
    ax.plot(sensitivity_df["sample_size"], sensitivity_df[col],
            marker="o", linewidth=2, color=color)
    ax.set_xlabel("Training sample size")
    ax.set_title(title, fontweight="bold")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
    ax.grid(alpha=0.3)

fig.suptitle("Sample sensitivity analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

## UMAP / HDBSCAN grid search

A systematic grid search identifies the optimal parameter combination before scaling BERTopic to the full corpus. 

Each parameter combination is evaluated by fitting a BERTopic model on the provided documents and embeddings. After every run, a checkpoint is saved so the search can be safely interrupted and resumed without losing progress.


### Metrics Recorded Per Combination

| Metric | Description |
|---|---|
| `silhouette` | Cluster separation in embedding space (excl. outlier documents).|
| `npmi_coherence` | Mean NPMI coherence across topic word pairs. |
| `num_topics` | Total number of discovered topics. |
| `outlier_pct` | Percentage of documents assigned to the outlier topic (`-1`). |
| `top5_topics` | Summary of the 5 largest topics and their top 8 representative words. |

### Parameters

| Parameter | Description |
|---|---|
| `docs` | List of documents used for topic modeling. |
| `embeddings` | Pre-computed embeddings corresponding to the documents. |
| `param_grid` | Dictionary containing the hyperparameter search space. |
| `checkpoint_path` | File path used to store checkpoint files during the search. |
| `save_csv_path` | File path for saving the final grid search results as a CSV file. |
| `verbose` | Whether to print progress information for each parameter combination. |

### Returns

| Output | Description |
|---|---|
| `grid_results_df` | DataFrame containing all evaluated parameter combinations. |
| `best_params` | Dictionary containing the best-performing parameter combination. |


### Checkpointing Behavior

- A checkpoint is written after every evaluated parameter combination.
- Interrupted runs can be resumed automatically from the latest saved checkpoint.

In [21]:
# Qualitative investigation of topics
def _print_sample_topics(topic_model: BERTopic, n_topics: int = 5) -> None:
    """Print top words and document count for a sample of topics."""
    info = topic_model.get_topic_info()
    info = info[info["Topic"] != -1].head(n_topics)
    for _, row in info.iterrows():
        words = ", ".join(w for w, _ in topic_model.get_topic(row["Topic"])[:8])
        print(f"    Topic {row['Topic']:>3} ({row['Count']:>5} docs): {words}")

# # Return the top-5 topic labels as a pipe-separated string for CSV storage.
# Each entry is 'TopicID: word1, word2, ..., word8 (N docs)'.
# Stored in the results CSV so qualitative review does not require refitting.
def _top5_topics_string(topic_model: BERTopic) -> str:
    info = topic_model.get_topic_info()
    info = info[info["Topic"] != -1].head(5)
    parts = []
    for _, row in info.iterrows():
        words = ", ".join(w for w, _ in topic_model.get_topic(row["Topic"])[:8])
        parts.append(f"T{row['Topic']}: {words} ({row['Count']} docs)")
    return " | ".join(parts)


def bertopic_grid_search(
    docs: list,
    embeddings: np.ndarray,
    param_grid: dict,
    checkpoint_path=CHECKPOINT_PATH_GRIDSEARCH,
    save_csv_path=FINAL_CSV_PATH_GRIDSEARCH,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict]:
    
    start_time = time.time()

    results = []
    checkpoint = safe_load_checkpoint(checkpoint_path)
    if checkpoint:
        results      = checkpoint.get("results", [])
        start_index  = checkpoint.get("last_index", -1) + 1
        if verbose:
            print(f"✅  Checkpoint loaded – resuming from combination #{start_index}")
    else:
        start_index = 0

    # Retrieve parameters
    param_combinations = list(product(
        param_grid["umap__n_neighbors"],
        param_grid["umap__n_components"],
        param_grid["umap__min_dist"],
        param_grid["hdbscan__min_cluster_size"],
        param_grid["hdbscan__min_samples"],
    ))
    total = len(param_combinations)
    if verbose:
        print(f"🔍  Total combinations: {total}")

    for idx, (n_neighbors, n_components, min_dist, min_cluster_size, min_samples) in \
            enumerate(param_combinations[start_index:], start=start_index):

        if verbose:
            print(
                f"\n🔹  [{idx+1}/{total}] neighbors={n_neighbors}, comp={n_components}, "
                f"dist={min_dist}, cluster={min_cluster_size}, samples={min_samples}"
            )
        # Run BERTopic on each combination
        try:
            umap_model = UMAP(
                n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist,
                metric="cosine", low_memory=True, random_state=42, n_jobs=-1,
            )
            hdbscan_model = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size, min_samples=min_samples,
                metric="euclidean", cluster_selection_method="eom",
                core_dist_n_jobs=-1, prediction_data=False,
            )
            vectorizer_model = CountVectorizer(
                ngram_range=(1, 1), stop_words="english",
                max_features=10_000, min_df=1,
            )
            topic_model = BERTopic(
                embedding_model=None,
                umap_model=umap_model,
                hdbscan_model=hdbscan_model,
                vectorizer_model=vectorizer_model,
                calculate_probabilities=False,
                verbose=False,
            )

            topics, _ = topic_model.fit_transform(docs, embeddings)
            topics_arr = np.array(topics)

            if verbose:
                print("  Sample topics:")
                _print_sample_topics(topic_model, n_topics=5)

            # Silhouette score
            non_outlier_mask = topics_arr != -1 # exclude outlier topic -1
            emb_filtered     = embeddings[non_outlier_mask]
            top_filtered     = topics_arr[non_outlier_mask]
            n_unique_real    = len(set(top_filtered))
            n_unique_total   = len(set(topics))  # includes -1
            outlier_pct      = round((~non_outlier_mask).sum() / len(topics_arr) * 100, 2)

            silhouette = (
                silhouette_score(emb_filtered, top_filtered)
                if n_unique_real > 1 else -1
            )

            # NPMI coherence
            npmi = compute_npmi_coherence(topic_model, docs, topk=10)

            # Top-5 topics string
            top5_str = _top5_topics_string(topic_model)

            results.append({
                "n_neighbors":      n_neighbors,
                "n_components":     n_components,
                "min_dist":         min_dist,
                "min_cluster_size": min_cluster_size,
                "min_samples":      min_samples,
                "silhouette":       round(silhouette, 6),
                "npmi_coherence":   round(npmi, 6),
                "num_topics":       n_unique_total,
                "outlier_pct":      outlier_pct,
                "top5_topics":      top5_str,
            })

            if verbose:
                print(
                    f"  ✅  Topics={n_unique_total} | Outliers={outlier_pct:.1f}% | "
                    f"Silhouette={silhouette:.4f} | NPMI={npmi:.4f}"
                )

        except Exception as e:
            if verbose:
                print(f"  ❌  Failed: {e}")
            results.append({
                "n_neighbors":      n_neighbors,
                "n_components":     n_components,
                "min_dist":         min_dist,
                "min_cluster_size": min_cluster_size,
                "min_samples":      min_samples,
                "silhouette":       np.nan,
                "npmi_coherence":   np.nan,
                "num_topics":       np.nan,
                "outlier_pct":      np.nan,
                "top5_topics":      "",
            })

        # Save rolling checkpoint
        ckpt = {"results": results, "last_index": idx}
        with open(checkpoint_path, "wb") as f:
            pickle.dump(ckpt, f, protocol=pickle.HIGHEST_PROTOCOL)
        if verbose:
            print("💾 Checkpoint saved.")

    grid_results_df = pd.DataFrame(results).sort_values(by="silhouette", ascending=False)
    best_params     = grid_results_df.iloc[0].to_dict()
    grid_results_df.to_csv(save_csv_path, index=False)

    elapsed = time.time() - start_time
    if verbose:
        print(f"\n🏁  Grid search complete in {elapsed/60:.1f} min")
        print(f"\nBest configuration:\n{best_params}")

    return grid_results_df, best_params

Set your hyperparameters below.

Parameter definitions to guide your decisions can be found [here](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html).

In [22]:
# CPU-friendly version (32 combinations)
param_grid = {
    "umap__n_neighbors":       [15, 30],
    "umap__n_components":      [5, 10],
    "umap__min_dist":          [0.0, 0.1],
    "hdbscan__min_cluster_size": [30, 100],
    "hdbscan__min_samples":    [15, 25],
}

In [ ]:
# # ⚠️ Runtime warning:
# E.g., up to ~7h on a GPU for 324 combinations with a 50k sample.
# Checkpoints ensure safe interruption.
grid_results_df, best_params = bertopic_grid_search(
    docs=docs_sample,
    embeddings=embeddings_sample,
    param_grid=param_grid,
    checkpoint_path=CHECKPOINT_PATH_GRIDSEARCH,
    save_csv_path=FINAL_CSV_PATH_GRIDSEARCH,
    verbose=True,
)

In [ ]:
print("Top-10 parameter combinations by silhouette score:")
grid_results_df.head(10)

In [ ]:
# Reload grid search results
grid_results_df = pd.read_csv(FINAL_CSV_PATH_GRIDSEARCH)
best_params     = grid_results_df.iloc[0].to_dict()
print(f"Best parameters:\n{best_params}")

### Save `best_params` as JSON with provenance metadata

Storing best parameters as a JSON file (rather than re-parsing the CSV each time) makes loading them in `3_topic_modelling.ipynb` trivial and keeps the chain complete. The file includes the embedding model name, embedding shape, and the sample size the grid search was run on, so the modelling notebook can verify it is using the correct inputs.

In [ ]:
# # Assemble metadata alongside the parameter values
# Strip non-parameter keys that came from the CSV (metrics, top5 string, etc.)
param_keys = [
    "n_neighbors", "n_components", "min_dist",
    "min_cluster_size", "min_samples",
]
best_params_clean = {
    # Cast floats that were read back from CSV as float64 → native Python types
    k: int(best_params[k]) if k != "min_dist" else float(best_params[k])
    for k in param_keys
}

provenance = {
    "best_params": best_params_clean,
    "grid_search_metrics": {
        "silhouette":     round(float(best_params["silhouette"]),     6),
        "npmi_coherence": round(float(best_params["npmi_coherence"]), 6),
        "num_topics":     int(best_params["num_topics"]),
        "outlier_pct":    float(best_params["outlier_pct"]),
        "top5_topics":    str(best_params.get("top5_topics", "")),
    },
    "provenance": {
        "embedding_model":  best_model_name,
        "embedding_shape":  list(embeddings_sample.shape),
        "gridsearch_sample_size": len(docs_sample),
        "gridsearch_csv":   str(FINAL_CSV_PATH_GRIDSEARCH),
    },
}

with open(BEST_PARAMS_PATH, "w") as f:
    json.dump(provenance, f, indent=2)

print(f"✅  Saved best_params + provenance.")

---
## ✅ Summary & next steps

The outputs of this notebook feed directly into `3_topic_modelling.ipynb`:

| Output | Description | Variable in next notebook |
|--------|-------------|---------------------------|
| `final_embeddings.npy` | Full-corpus embeddings | `embeddings` |
| `gridsearch_results.csv` | All parameter combinations + silhouette, NPMI, outlier %, top-5 topics | `grid_results_df` |
| `best_params.json` | Optimal parameters + provenance metadata | loaded via `json.load` |

---
# AI disclosure statement

AI tools were used to assist:
- developing, labelling, and debugging code
- formatting Markdown cells

AI tools used:
- [CursorAI (Desktop version)](https://cursor.com/agents)
- [Claude AI](https://claude.ai/)
- [ChatGPT](https://chatgpt.com/)

I acknowledge my responsibility as a researcher to thoroughly verify all outputs and content produced by AI tools and accept full accountability for their accuracy and validity.

XXX

# References
- Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure. [arXiv preprint arXiv:2203.05794](https://arxiv.org/abs/2203.05794).
- Van Atteveldt, W., Trilling, D., & Calderón, C. A. (2022). Computational Analysis of Communication. Wiley Blackwell. https://cssbook.net/